# 00 - Submission Creator & Results Aggregator

This notebook:
1. Collects all prediction results from inference notebooks
2. Creates properly formatted JSON submissions for CodaBench
3. Tracks model/technique names for Task 2
4. Syncs results to Google Drive

**Expected Output Format:**
```json
[
  {"id": "utt_00123", "text_diacritized": "النص المُشَكَّل هنا"},
  {"id": "utt_00124", "text_diacritized": "هَذا نَصٌّ مُشَكَّلٌ آخَر"}
]
```

In [ ]:
# Setup - Import from config.py (generated by setup.sh)
import os
import sys
import json
import zipfile
import pandas as pd
from datetime import datetime
from pathlib import Path
from collections import defaultdict

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR, DEVICE,
        TRAIN_AUDIO_DIR, TRAIN_TEXT_DIR, TRAIN_LIST,
        DEV_AUDIO_DIR, DEV_TEXT_DIR, DEV_LIST,
        DEV_INPUT, DEV_OUTPUT,
        TEST_DIR, TEST_AUDIO_DIR, TEST_LIST,
        OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINT_DIR
    )
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

print(f"Environment: 'Local'")
print(f"Project: {PROJECT_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Test: {TEST_DIR}")

## Model Registry

Track all models/techniques used for submissions

In [ ]:
# Model/Technique Registry for Task 2
MODEL_REGISTRY = {
    'fine_tashkeel': {
        'name': 'Fine-Tashkeel',
        'hf_path': 'basharalrfooh/Fine-Tashkeel',
        'architecture': 'ByT5 (Byte-level Seq2Seq)',
        'technique': 'Byte-level text-to-text generation'
    },
    'mushkil': {
        'name': 'Mushkil',
        'hf_path': 'riotu-lab/mushkil',
        'architecture': 'AraT5v2 Seq2Seq',
        'technique': 'Arabic T5 text generation'
    },
    'flan_t5': {
        'name': 'FLAN-T5 Arabic Tashkeel',
        'hf_path': 'Abdou/arabic-tashkeel-flan-t5-small',
        'architecture': 'FLAN-T5 Seq2Seq',
        'technique': 'Instruction-tuned T5'
    },
    'catt': {
        'name': 'CATT',
        'hf_path': 'abjadai/catt',
        'architecture': 'Character-level Transformer',
        'technique': 'Character-based token classification'
    },
    'arat5': {
        'name': 'AraT5-base',
        'hf_path': 'UBC-NLP/AraT5-base',
        'architecture': 'T5 Encoder-Decoder',
        'technique': 'Arabic-specific T5 generation'
    },
    'arabert': {
        'name': 'AraBERT',
        'hf_path': 'aubmindlab/bert-base-arabertv2',
        'architecture': 'BERT Token Classification',
        'technique': 'Token-level diacritic prediction'
    },
    'marbert': {
        'name': 'MARBERT',
        'hf_path': 'UBC-NLP/MARBERT',
        'architecture': 'BERT Encoder',
        'technique': 'Multi-dialect Arabic understanding'
    },
    'sadeed': {
        'name': 'Sadeed',
        'hf_path': 'QCRI/Sadeed',
        'architecture': 'Small LLM Decoder',
        'technique': 'Decoder-only generation'
    },
    'whisper_multimodal': {
        'name': 'Whisper + Diacritization Pipeline',
        'hf_path': 'openai/whisper-large-v3',
        'architecture': 'Speech Encoder + Text Diacritizer',
        'technique': 'Multimodal audio-guided diacritization'
    },
    'gemma': {
        'name': 'Gemma Arabic',
        'hf_path': 'google/gemma-2-9b',
        'architecture': 'Decoder-only LLM',
        'technique': 'Instruction-following diacritization'
    },
    'mt5': {
        'name': 'mT5-base',
        'hf_path': 'google/mt5-base',
        'architecture': 'Multilingual T5',
        'technique': 'Multilingual seq2seq generation'
    },
    'jais': {
        'name': 'Jais-13B',
        'hf_path': 'core42/jais-13b',
        'architecture': 'Decoder-only Arabic LLM',
        'technique': 'Arabic LLM generation'
    },
    'ensemble_voting': {
        'name': 'Ensemble (Majority Voting)',
        'hf_path': 'N/A - Ensemble',
        'architecture': 'Multi-model Ensemble',
        'technique': 'Character-level majority voting'
    },
    'ensemble_weighted': {
        'name': 'Ensemble (Weighted)',
        'hf_path': 'N/A - Ensemble',
        'architecture': 'Multi-model Ensemble',
        'technique': 'WER-weighted character voting'
    },
    'postprocess': {
        'name': 'Post-processed Output',
        'hf_path': 'N/A - Post-processing',
        'architecture': 'Rule-based refinement',
        'technique': 'Linguistic rules + spell correction'
    }
}

print(f"Registered {len(MODEL_REGISTRY)} models/techniques")

## Collect All Predictions

In [ ]:
# Scan for all prediction files
dev_predictions = {}
test_predictions = {}

for pred_file in OUTPUT_DIR.glob('*_dev_predictions.json'):
    model_key = pred_file.stem.replace('_dev_predictions', '')
    with open(pred_file, 'r', encoding='utf-8') as f:
        dev_predictions[model_key] = json.load(f)
    print(f"[DEV] Loaded: {model_key} ({len(dev_predictions[model_key])} samples)")

for pred_file in OUTPUT_DIR.glob('*_test_predictions.json'):
    model_key = pred_file.stem.replace('_test_predictions', '')
    with open(pred_file, 'r', encoding='utf-8') as f:
        test_predictions[model_key] = json.load(f)
    print(f"[TEST] Loaded: {model_key} ({len(test_predictions[model_key])} samples)")

print(f"\nTotal: {len(dev_predictions)} dev, {len(test_predictions)} test prediction files")

## Evaluation Metrics

In [ ]:
import re
from tqdm import tqdm

DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}  # tanween

def extract_diacritics(text):
    """Extract diacritics for each Arabic letter."""
    result = []
    i = 0
    while i < len(text):
        char = text[i]
        if ARABIC_LETTER_PATTERN.match(char):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((char, ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(pred_text, ref_text, exclude_irab=False):
    """Compute DER, WER, SER for a single sample."""
    pred_pairs = extract_diacritics(pred_text)
    ref_pairs = extract_diacritics(ref_text)
    
    # DER
    min_len = min(len(pred_pairs), len(ref_pairs))
    char_errors = 0
    for i in range(min_len):
        pred_d = pred_pairs[i][1]
        ref_d = ref_pairs[i][1]
        if exclude_irab:
            pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
            ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
        if pred_d != ref_d:
            char_errors += 1
    char_errors += abs(len(pred_pairs) - len(ref_pairs))
    total_chars = max(len(pred_pairs), len(ref_pairs))
    
    # WER
    pred_words = pred_text.split()
    ref_words = ref_text.split()
    word_errors = 0
    for i in range(min(len(pred_words), len(ref_words))):
        if pred_words[i] != ref_words[i]:
            word_errors += 1
    word_errors += abs(len(pred_words) - len(ref_words))
    total_words = max(len(pred_words), len(ref_words))
    
    # SER
    ser = 1.0 if char_errors > 0 else 0.0
    
    return {
        'der': char_errors / total_chars if total_chars > 0 else 0,
        'wer': word_errors / total_words if total_words > 0 else 0,
        'ser': ser,
        'char_errors': char_errors,
        'word_errors': word_errors,
        'total_chars': total_chars,
        'total_words': total_words
    }

def evaluate_predictions(predictions, ground_truth, exclude_irab=False):
    """Evaluate all predictions against ground truth."""
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    
    total = {'chars': 0, 'words': 0, 'char_errors': 0, 'word_errors': 0, 'ser': 0}
    n = 0
    
    for pred in predictions:
        if pred['id'] not in gt_lookup:
            continue
        metrics = compute_metrics(pred['text_diacritized'], gt_lookup[pred['id']], exclude_irab)
        total['chars'] += metrics['total_chars']
        total['words'] += metrics['total_words']
        total['char_errors'] += metrics['char_errors']
        total['word_errors'] += metrics['word_errors']
        total['ser'] += metrics['ser']
        n += 1
    
    return {
        'DER': total['char_errors'] / total['chars'] if total['chars'] > 0 else 0,
        'WER': total['word_errors'] / total['words'] if total['words'] > 0 else 0,
        'SER': total['ser'] / n if n > 0 else 0,
        'n_samples': n
    }

## Evaluate Dev Predictions

In [ ]:
# Load ground truth
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

# Evaluate all dev predictions
results = []

for model_key, predictions in dev_predictions.items():
    print(f"\nEvaluating: {model_key}")
    
    # With I'rab
    metrics_with = evaluate_predictions(predictions, ground_truth, exclude_irab=False)
    
    # Without I'rab
    metrics_no = evaluate_predictions(predictions, ground_truth, exclude_irab=True)
    
    model_info = MODEL_REGISTRY.get(model_key, {'name': model_key, 'technique': 'Unknown'})
    
    results.append({
        'Model': model_info['name'],
        'Technique': model_info.get('technique', ''),
        'WER (w/ Irab)': f"{metrics_with['WER']*100:.2f}%",
        'DER (w/ Irab)': f"{metrics_with['DER']*100:.2f}%",
        'SER (w/ Irab)': f"{metrics_with['SER']*100:.2f}%",
        'WER (no Irab)': f"{metrics_no['WER']*100:.2f}%",
        'DER (no Irab)': f"{metrics_no['DER']*100:.2f}%",
        'Samples': metrics_with['n_samples']
    })
    
    print(f"  WER: {metrics_with['WER']*100:.2f}% (w/ Irab) | {metrics_no['WER']*100:.2f}% (no Irab)")

# Summary table
if results:
    df = pd.DataFrame(results)
    print("\n" + "="*100)
    print("DEV SET RESULTS SUMMARY")
    print("="*100)
    print(df.to_string(index=False))

## Create Blind Test Submissions

In [ ]:
def create_submission(predictions, model_key, submission_dir):
    """
    Create a CodaBench-compatible submission.
    
    Output format:
    [{"id": "...", "text_diacritized": "..."}]
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    model_info = MODEL_REGISTRY.get(model_key, {'name': model_key})
    
    # Ensure correct format
    formatted = []
    for item in predictions:
        formatted.append({
            'id': item['id'],
            'text_diacritized': item['text_diacritized']
        })
    
    # Save JSON
    json_file = submission_dir / f"{model_key}_submission.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(formatted, f, ensure_ascii=False, indent=2)
    
    # Create ZIP
    zip_file = submission_dir / f"{model_key}_submission_{timestamp}.zip"
    with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_file, 'submission.json')
    
    return {
        'model': model_key,
        'name': model_info['name'],
        'json_file': str(json_file),
        'zip_file': str(zip_file),
        'size_kb': zip_file.stat().st_size / 1024,
        'samples': len(formatted)
    }

In [ ]:
# Create all blind test submissions
submissions_created = []

for model_key, predictions in test_predictions.items():
    print(f"Creating submission: {model_key}")
    result = create_submission(predictions, model_key, SUBMISSION_DIR)
    submissions_created.append(result)
    print(f"  -> {result['zip_file']} ({result['size_kb']:.1f} KB, {result['samples']} samples)")

print(f"\nCreated {len(submissions_created)} submissions")

## Submissions Summary

In [ ]:
# List all submissions ready for upload
print("="*80)
print("SUBMISSIONS READY FOR CODABENCH")
print("="*80)

for sub in submissions_created:
    print(f"\n{sub['name']}")
    print(f"  File: {sub['zip_file']}")
    print(f"  Size: {sub['size_kb']:.1f} KB")
    print(f"  Samples: {sub['samples']}")

## Sync to Google Drive

In [ ]:
# Google Drive sync removed for public release


## Export Results Report

In [ ]:
# Save comprehensive results report
report = {
    'timestamp': datetime.now().isoformat(),
    'dev_results': results,
    'submissions': submissions_created,
    'model_registry': MODEL_REGISTRY
}

report_file = OUTPUT_DIR / 'evaluation_report.json'
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Report saved: {report_file}")